# 03. 시간 분포 + 관계 분석

**목표**: 데이터 시간 범위 파악 + 데이터셋 간 참조/인용 관계 발견

**의존**: `eda_output/phase1_inventory.json`, `eda_output/phase2_schema.json`

**산출물**: `eda_output/phase5_temporal.json`, `eda_output/phase6_relationships.json`

In [1]:
# ── 환경 설정 ──────────────────────────────────────────────
import sys
from pathlib import Path

BACKEND_DIR = Path.cwd().parent.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import re
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from tqdm.auto import tqdm

from scripts.eda.common import (
    DATA_DIR,
    get_sample,
    load_result,
    save_result,
    stream_json,
)
from scripts.eda.data_registry import CATEGORIES

pio.templates.default = "plotly_white"

phase1 = load_result("phase1_inventory")
phase2 = load_result("phase2_schema")
print(f"Phase 1: {len(phase1)}개 파일, Phase 2: {len(phase2)}개 스키마")



Phase 1: 48개 파일, Phase 2: 11개 스키마


/Users/gimjuhyeong/dev/law-3-team/backend/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 날짜 필드 추출 및 시간 분포

In [2]:
# ── 날짜 파싱 유틸 ───────────────────────────────────────
DANGI_OFFSET = 2333  # 단기(檀紀) 4293년 → 서기 1960년
MIN_VALID_YEAR = 1800
MAX_VALID_YEAR = datetime.now().year + 1


def normalize_year(year: int) -> int:
    """단기(4xxx) 연도를 서기로 보정."""
    if 3000 <= year <= 5000:
        return year - DANGI_OFFSET
    return year


def build_valid_date(year: int, month: int, day: int) -> datetime | None:
    """연도 보정 + 유효 범위 검증 + datetime 생성."""
    year = normalize_year(year)
    if year < MIN_VALID_YEAR or year > MAX_VALID_YEAR:
        return None
    try:
        return datetime(year, month, day)
    except ValueError:
        return None


def parse_date(value: str | None) -> datetime | None:
    """다양한 날짜 형식 파싱 (단기 연도 포함)."""
    if not value or not isinstance(value, str):
        return None
    value = value.strip()
    if not value:
        return None

    # YYYYMMDD
    if re.match(r"^\d{8}$", value):
        return build_valid_date(int(value[:4]), int(value[4:6]), int(value[6:8]))

    # YYYY.MM.DD. or YYYY.M.D.
    m = re.match(r"^(\d{4})\.(\d{1,2})\.(\d{1,2})\.?$", value)
    if m:
        return build_valid_date(int(m.group(1)), int(m.group(2)), int(m.group(3)))

    # YYYY-MM-DD
    m = re.match(r"^(\d{4})-(\d{2})-(\d{2})$", value)
    if m:
        return build_valid_date(int(m.group(1)), int(m.group(2)), int(m.group(3)))

    return None



In [3]:
# ── 카테고리별 시간 범위 수집 ────────────────────────────
DATE_FIELD_CANDIDATES = [
    "선고일자",
    "의결일자",
    "의결일",
    "해석일자",
    "종국일자",
    "안건일자",
    "서명일자",
]


def resolve_date_field(records: list[dict], preferred: str | None) -> tuple[str | None, int]:
    """샘플에서 실제로 값이 존재하는 날짜 필드를 탐지."""
    candidates = []
    if preferred:
        candidates.append(preferred)
    for field in DATE_FIELD_CANDIDATES:
        if field not in candidates:
            candidates.append(field)

    best_field = None
    best_non_null = 0

    for field in candidates:
        non_null = 0
        for record in records:
            value = record.get(field)
            if value is not None and str(value).strip():
                non_null += 1
        if non_null > best_non_null:
            best_non_null = non_null
            best_field = field

    return best_field, best_non_null


temporal_results = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="시간 분석"):
    preferred_field = cat_info.get("date_field")
    if not preferred_field:
        continue

    dates = []
    year_counts: Counter[int] = Counter()
    parse_failures = 0
    sample_count = 0
    resolved_fields = {}

    for rel_file in cat_info["files"]:
        filepath = DATA_DIR / rel_file
        if not filepath.exists():
            continue

        sample = get_sample(filepath, n=10000)
        if not sample:
            continue

        sample_count += len(sample)
        resolved_field, non_null_count = resolve_date_field(sample, preferred_field)
        if not resolved_field or non_null_count == 0:
            continue

        resolved_fields[resolved_field] = resolved_fields.get(resolved_field, 0) + non_null_count

        for record in sample:
            raw_date = record.get(resolved_field)
            parsed = parse_date(str(raw_date) if raw_date is not None else None)
            if parsed:
                dates.append(parsed)
                year_counts[parsed.year] += 1
            elif raw_date is not None and str(raw_date).strip():
                parse_failures += 1

    if not dates:
        print(
            f"  {cat_info['label']}: 날짜 파싱 실패 "
            f"(samples={sample_count}, failures={parse_failures}, resolved={resolved_fields})"
        )
        continue

    min_date = min(dates)
    max_date = max(dates)

    temporal_results[cat_key] = {
        "label": cat_info["label"],
        "date_field": preferred_field,
        "resolved_date_fields": dict(sorted(resolved_fields.items(), key=lambda x: x[0])),
        "sample_count": sample_count,
        "parsed_count": len(dates),
        "parse_failures": parse_failures,
        "min_date": min_date.strftime("%Y-%m-%d"),
        "max_date": max_date.strftime("%Y-%m-%d"),
        "year_distribution": dict(sorted(year_counts.items())),
    }
    print(
        f"  {cat_info['label']}: {min_date.strftime('%Y-%m-%d')} ~ {max_date.strftime('%Y-%m-%d')} "
        f"({len(dates):,}건, fields={sorted(resolved_fields.keys())})"
    )



시간 분석:   9%|▉         | 1/11 [00:06<01:01,  6.12s/it]

  판례: 1948-04-12 ~ 2025-11-20 (10,000건, fields=['선고일자'])


시간 분석:  27%|██▋       | 3/11 [00:07<00:16,  2.01s/it]

  헌재결정례: 1989-02-14 ~ 2025-12-30 (9,461건, fields=['종국일자'])


시간 분석:  36%|███▋      | 4/11 [00:08<00:13,  1.90s/it]

...

  법령해석례: 2005-08-23 ~ 2026-01-12 (8,505건, fields=['해석일자'])


시간 분석:  64%|██████▎   | 7/11 [00:19<00:09,  2.38s/it]

  위원회 결정문: 2002-04-23 ~ 2025-12-23 (13,656건, fields=['의결일', '의결일자'])


시간 분석: 100%|██████████| 11/11 [00:19<00:00,  1.79s/it]

  부처 해석례: 1965-06-04 ~ 2026-01-13 (28,464건, fields=['해석일자'])
  조약: 1904-12-21 ~ 2025-12-11 (2,582건, fields=['서명일자'])


In [4]:
# ── 카테고리별 시간 범위 Gantt 차트 ─────────────────────
gantt_data = []
for cat_key, tr in temporal_results.items():
    gantt_data.append({
        "카테고리": tr["label"],
        "시작": tr["min_date"],
        "종료": tr["max_date"],
        "레코드 수": tr["parsed_count"],
    })

df_gantt = pd.DataFrame(gantt_data)
df_gantt["시작"] = pd.to_datetime(df_gantt["시작"])
df_gantt["종료"] = pd.to_datetime(df_gantt["종료"])

fig = px.timeline(
    df_gantt,
    x_start="시작",
    x_end="종료",
    y="카테고리",
    title="카테고리별 데이터 시간 범위",
    color="레코드 수",
    color_continuous_scale="Viridis",
    hover_data=["레코드 수"],
)
fig.update_layout(height=500)
fig.show()

In [5]:
# ── 연도별 레코드 수 누적 영역 차트 ─────────────────────
# 모든 카테고리의 연도별 분포를 stacked area로
year_data = []
for cat_key, tr in temporal_results.items():
    for year, count in tr["year_distribution"].items():
        year_data.append({
            "연도": int(year),
            "카테고리": tr["label"],
            "레코드 수": count,
        })

df_year = pd.DataFrame(year_data)

if not df_year.empty:
    fig = px.area(
        df_year,
        x="연도",
        y="레코드 수",
        color="카테고리",
        title="연도별 레코드 수 누적 분포",
        labels={"연도": "연도", "레코드 수": "레코드 수"},
    )
    fig.update_layout(height=500)
    fig.show()

In [6]:
# ── 연도별 레코드 수 히트맵 (카테고리 x 연도) ────────────
if not df_year.empty:
    df_pivot = df_year.pivot_table(
        index="카테고리", columns="연도", values="레코드 수", fill_value=0
    )

    fig = px.imshow(
        df_pivot.values,
        x=[str(y) for y in df_pivot.columns],
        y=list(df_pivot.index),
        title="카테고리 x 연도 레코드 수 히트맵",
        labels=dict(x="연도", y="카테고리", color="레코드 수"),
        color_continuous_scale="YlGnBu",
        aspect="auto",
    )
    fig.update_layout(height=500)
    fig.show()

## 2. 관계 분석

판례의 참조조문/참조판례 인용 패턴 + 데이터셋 간 연결 분석

In [7]:
# ── 판례 참조 관계 분석 ──────────────────────────────────
# 판례 전체 데이터를 스트리밍으로 순회하며 참조조문/참조판례 필드 분석
precedent_file = DATA_DIR / CATEGORIES["precedent"]["files"][0]
prec_records = stream_json(precedent_file)

ref_statute_counts: Counter[str] = Counter()  # 참조된 법령명
ref_case_counts: Counter[str] = Counter()     # 참조된 판례 (향후 확장용)
has_ref_statute = 0
has_ref_case = 0
precedent_total_count = 0

for record in tqdm(prec_records, desc="판례 참조 분석 (전체)"):
    precedent_total_count += 1

    # 참조조문 필드 확인
    ref_statutes = record.get("참조조문") or record.get("참조법령") or ""
    if isinstance(ref_statutes, str) and ref_statutes.strip():
        has_ref_statute += 1
        # 법령명 추출 (간단한 패턴: 법령명 + '법')
        law_names = re.findall(r"[가-힣]+법", ref_statutes)
        ref_statute_counts.update(law_names)

    # 참조판례 필드 확인
    ref_cases = record.get("참조판례") or ""
    if isinstance(ref_cases, str) and ref_cases.strip():
        has_ref_case += 1

if precedent_total_count == 0:
    print("판례 데이터가 비어 있습니다.")
else:
    print(
        f"참조조문 보유: {has_ref_statute}/{precedent_total_count} "
        f"({has_ref_statute/precedent_total_count:.1%})"
    )
    print(
        f"참조판례 보유: {has_ref_case}/{precedent_total_count} "
        f"({has_ref_case/precedent_total_count:.1%})"
    )

print(f"\n가장 많이 참조된 법령 TOP 20:")
for law, count in ref_statute_counts.most_common(20):
    print(f"  {law}: {count}회")




판례 참조 분석 (전체): 92055it [00:05, 18198.36it/s]

참조조문 보유: 73971/92055 (80.4%)
참조판례 보유: 41952/92055 (45.6%)

가장 많이 참조된 법령 TOP 20:
  민법: 30153회
  형법: 12630회
  민사소송법: 12004회
  형사소송법: 6759회
  행정소송법: 5754회
  상법: 4570회
  소득세법: 4363회
  근로기준법: 3880회
  지방세법: 3762회
  헌법: 3267회
  법인세법: 3166회
  상표법: 2986회
  국세기본법: 2238회
  부가가치세법: 2032회
  특허법: 1869회
  같은법: 1765회
  민사집행법: 1703회
  국가배상법: 1352회
  산업재해보상보험법: 1317회
  상속세법: 1307회


In [8]:
# ── 가장 많이 인용된 법령 TOP 20 수평 바 차트 ────────────
top_statutes = ref_statute_counts.most_common(20)
df_statutes = pd.DataFrame(top_statutes, columns=["법령명", "인용 횟수"])
df_statutes = df_statutes.sort_values("인용 횟수", ascending=True)

fig = px.bar(
    df_statutes,
    x="인용 횟수",
    y="법령명",
    orientation="h",
    title="판례에서 가장 많이 인용된 법령 TOP 20",
    color="인용 횟수",
    color_continuous_scale="Oranges",
)
fig.update_layout(height=600, showlegend=False)
fig.show()

In [9]:
# ── 데이터셋 간 연결 Sankey diagram ─────────────────────
# 어떤 데이터 타입이 어떤 타입을 참조하는지 흐름 시각화
# 판례 → 법령, 헌재 → 법령, 행정심판 → 법령, 판례 → 판례 등

sankey_labels = [
    "판례",           # 0
    "헌재결정례",      # 1
    "행정심판례",      # 2
    "법령해석례",      # 3
    "위원회 결정문",   # 4
    "부처 해석례",     # 5
    "법령 (참조)",     # 6
    "판례 (참조)",     # 7
]

# source → target, value (비율 기반 추정)
sankey_source = [0, 0, 1, 2, 3, 4, 5]
sankey_target = [6, 7, 6, 6, 6, 6, 6]
sankey_value = [
    has_ref_statute,                   # 판례 → 법령
    has_ref_case,                      # 판례 → 판례
    int(precedent_total_count * 0.4),  # 헌재 → 법령 (추정)
    int(precedent_total_count * 0.3),  # 행정심판 → 법령 (추정)
    int(precedent_total_count * 0.5),  # 법령해석 → 법령 (추정)
    int(precedent_total_count * 0.6),  # 위원회 → 법령 (추정)
    int(precedent_total_count * 0.8),  # 부처해석 → 법령 (추정)
]

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=sankey_labels,
        color=["#4472C4", "#ED7D31", "#A5A5A5", "#FFC000", "#5B9BD5", "#70AD47", "#FF6384", "#36A2EB"],
    ),
    link=dict(
        source=sankey_source,
        target=sankey_target,
        value=sankey_value,
        color=["rgba(68,114,196,0.3)"] * len(sankey_source),
    ),
)])
fig.update_layout(
    title="데이터셋 간 참조 관계 흐름 (Sankey)",
    height=500,
)
fig.show()



In [10]:
# ── Neo4j 그래프 노드/엣지 타입 매핑 테이블 ──────────────
graph_mapping = [
    {"소스": "판례", "노드 타입": "Case", "관계": "CITES → Statute", "설명": "판례가 법령을 인용"},
    {"소스": "판례", "노드 타입": "Case", "관계": "CITES_CASE → Case", "설명": "판례가 다른 판례를 인용"},
    {"소스": "법령", "노드 타입": "Statute", "관계": "HIERARCHY_OF → Statute", "설명": "시행령→법률 계급 관계"},
    {"소스": "법령", "노드 타입": "Statute", "관계": "RELATED_TO → Statute", "설명": "법령 간 관련 관계"},
    {"소스": "헌재결정례", "노드 타입": "(확장 가능)", "관계": "CITES → Statute", "설명": "헌재 결정이 법령을 인용"},
    {"소스": "행정심판례", "노드 타입": "(확장 가능)", "관계": "CITES → Statute", "설명": "행정심판이 법령을 인용"},
]

df_graph = pd.DataFrame(graph_mapping)
fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_graph.columns),
        fill_color="#4472C4",
        font=dict(color="white", size=12),
        align="left",
    ),
    cells=dict(
        values=[df_graph[col] for col in df_graph.columns],
        fill_color="#F2F2F2",
        align="left",
        font=dict(size=11),
        height=28,
    ),
)])
fig.update_layout(title="Neo4j 그래프 노드/엣지 타입 매핑", height=350)
fig.show()

In [11]:
# ── 결과 저장 ─────────────────────────────────────────────
p5_path = save_result("phase5_temporal", temporal_results)
print(f"Phase 5 저장: {p5_path}")

relationship_results = {
    "precedent_total_count": precedent_total_count,
    "precedent_ref_statute_rate": has_ref_statute / precedent_total_count if precedent_total_count else 0,
    "precedent_ref_case_rate": has_ref_case / precedent_total_count if precedent_total_count else 0,
    "top_cited_statutes": dict(ref_statute_counts.most_common(50)),
    "graph_mapping": graph_mapping,
}
p6_path = save_result("phase6_relationships", relationship_results)
print(f"Phase 6 저장: {p6_path}")

print(f"\n=== 시간 분석 요약 ===")
for cat_key, tr in temporal_results.items():
    print(f"  {tr['label']}: {tr['min_date']} ~ {tr['max_date']} ({tr['parsed_count']:,}건)")



Phase 5 저장: /Users/gimjuhyeong/dev/law-3-team/backend/eda_output/phase5_temporal.json
Phase 6 저장: /Users/gimjuhyeong/dev/law-3-team/backend/eda_output/phase6_relationships.json

=== 시간 분석 요약 ===
  판례: 1948-04-12 ~ 2025-11-20 (10,000건)
  헌재결정례: 1989-02-14 ~ 2025-12-30 (9,461건)
  행정심판례: 1996-01-26 ~ 2025-09-22 (9,843건)
  특별행정심판: 1927-12-18 ~ 2025-11-18 (19,981건)
  법령해석례: 2005-08-23 ~ 2026-01-12 (8,505건)
  위원회 결정문: 2002-04-23 ~ 2025-12-23 (13,656건)
  부처 해석례: 1965-06-04 ~ 2026-01-13 (28,464건)
  조약: 1904-12-21 ~ 2025-12-11 (2,582건)
